# Table of Contents

1. [Data Loading and Initial Exploration](#b8532810)
2. [Path Construction](#37d25dd4)
3. [Custom Dataset and Transforms](#847f4e12)
4. [Model Initialization](#28432a3d)
5. [Inference and Submission Generation](#9bf415f1)
6. [References](#867b8bba)

In [ ]:
!nvidia-smi

Tue Mar 10 03:41:19 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   38C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

### 1. Data Loading and Initial Exploration
In this step, we load the training metadata from `train.csv` and define the directory containing the unzipped images. We then display the first few rows to understand the structure of the dataset.

In [ ]:
!sudo apt-get install tree -qq

In [ ]:
!unzip /content/train_images.zip

Streaming output truncated to the last 5000 lines.
  inflating: train_images/a35b96f5-c0bb-4d91-b446-87d46ff0d579.jpeg  
  inflating: __MACOSX/train_images/._a35b96f5-c0bb-4d91-b446-87d46ff0d579.jpeg  
  inflating: train_images/17970c2c-1618-4220-a42d-d593b954c7bf.jpeg  
  inflating: __MACOSX/train_images/._17970c2c-1618-4220-a42d-d593b954c7bf.jpeg  
  inflating: train_images/c3427567-8b3d-44dc-85b9-a81dbefec4d9.jpeg  
  inflating: __MACOSX/train_images/._c3427567-8b3d-44dc-85b9-a81dbefec4d9.jpeg  
  inflating: train_images/8d5116c3-18a8-478c-a33a-1b521bba92cd.jpeg  
  inflating: __MACOSX/train_images/._8d5116c3-18a8-478c-a33a-1b521bba92cd.jpeg  
  inflating: train_images/802d4484-83ac-438f-bf72-07134008610d.jpeg  
  inflating: __MACOSX/train_images/._802d4484-83ac-438f-bf72-07134008610d.jpeg  
  inflating: train_images/6160a379-3a91-4d0e-84b3-e8a5052e865c.jpeg  
  inflating: __MACOSX/train_images/._6160a379-3a91-4d0e-84b3-e8a5052e865c.jpeg  
  inflating: train_images/eb2a112d-301a-4c5

In [ ]:
import pandas as pd
import os

# Load the dataset
df = pd.read_csv('/content/train.csv')

# Define the directory where images were unzipped
# Based on your previous unzip command, it created a 'train_images' folder
image_dir = '/content/train_images'

# Display the first few rows to see the column names
display(df.head())

,image_id,label
0,bc927c06-88af-469c-b00d-41a540e26860.jpeg,Moles
1,7eaabc41-efa1-4f3f-be2b-38abc1bc36a3.jpeg,Benign_tumors
2,41488eb8-bbfa-483c-bac6-c1f743eef13f.jpeg,Eczema
3,e1ca09ce-020e-4ab2-a18b-b2fca476ac47.jpeg,Seborrh_Keratoses
4,50eb7861-2bff-46dd-85dc-b63a5d3aaff5.jpeg,Vascular_Tumors


In [ ]:
# Assuming there is a column like 'image_id' or 'filename'
# We create a new column with the full system path
# Replace 'image_id' with the actual column name from the output above
if 'image_id' in df.columns:
    # Correctly construct file_path by using 'x' directly, as 'image_id' already contains the .jpeg extension
    df['file_path'] = df['image_id'].apply(lambda x: os.path.join(image_dir, x))
    display(df[['image_id', 'file_path']].head())
else:
    print("Please check the column names above and update 'image_id' to the correct reference.")

,image_id,file_path
0,bc927c06-88af-469c-b00d-41a540e26860.jpeg,/content/train_images/bc927c06-88af-469c-b00d-...
1,7eaabc41-efa1-4f3f-be2b-38abc1bc36a3.jpeg,/content/train_images/7eaabc41-efa1-4f3f-be2b-...
2,41488eb8-bbfa-483c-bac6-c1f743eef13f.jpeg,/content/train_images/41488eb8-bbfa-483c-bac6-...
3,e1ca09ce-020e-4ab2-a18b-b2fca476ac47.jpeg,/content/train_images/e1ca09ce-020e-4ab2-a18b-...
4,50eb7861-2bff-46dd-85dc-b63a5d3aaff5.jpeg,/content/train_images/50eb7861-2bff-46dd-85dc-...


### 2. Path Construction
To facilitate image loading during training, we construct the full system path for each image by appending the filename to the image directory path.

In [ ]:
from PIL import Image
from torch.utils.data import Dataset
import torchvision.transforms as transforms

class SkinDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.df = dataframe
        self.transform = transform
        # Create a mapping from label names to integers
        self.label_map = {label: idx for idx, label in enumerate(self.df['label'].unique())}

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img_path = self.df.iloc[idx]['file_path']
        image = Image.open(img_path).convert('RGB')
        label = self.label_map[self.df.iloc[idx]['label']]

        if self.transform:
            image = self.transform(image)

        return image, label

# Define the transformations for the images
transform = transforms.Compose([
    transforms.Resize((224, 224)), # Resize images to 224x224
    transforms.ToTensor(),         # Convert images to PyTorch tensors
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]) # Normalize image pixel values
])

# Initialize the dataset using the DataFrame we created earlier
dataset = SkinDataset(df, transform=transform)
print(f'Dataset created with {len(dataset)} samples.')

Dataset created with 8642 samples.


In [ ]:
!unzip /content/test_images.zip

Streaming output truncated to the last 5000 lines.
  inflating: test_images/6c534a57-c84d-45e5-8d4a-08ac22b5e3cf.jpeg  
  inflating: __MACOSX/test_images/._6c534a57-c84d-45e5-8d4a-08ac22b5e3cf.jpeg  
  inflating: test_images/c52de06f-c48f-4c67-9541-2377690579ae.jpeg  
  inflating: __MACOSX/test_images/._c52de06f-c48f-4c67-9541-2377690579ae.jpeg  
  inflating: test_images/de2eaaa3-d8fe-4041-8e8b-5c1583a6c6aa.jpeg  
  inflating: __MACOSX/test_images/._de2eaaa3-d8fe-4041-8e8b-5c1583a6c6aa.jpeg  
  inflating: test_images/86bd1c70-29e7-4f04-9fa3-0950226c4cab.jpeg  
  inflating: __MACOSX/test_images/._86bd1c70-29e7-4f04-9fa3-0950226c4cab.jpeg  
  inflating: test_images/475e28bb-ca7c-45c1-adf8-49a5b8c11f0c.jpeg  
  inflating: __MACOSX/test_images/._475e28bb-ca7c-45c1-adf8-49a5b8c11f0c.jpeg  
  inflating: test_images/fca61a5f-ae72-46d1-93d9-d848970512d1.jpeg  
  inflating: __MACOSX/test_images/._fca61a5f-ae72-46d1-93d9-d848970512d1.jpeg  
  inflating: test_images/673a6545-6669-4fd4-a341-285c68

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import transforms, datasets
from torch.utils.data import DataLoader
import timm
from tqdm import tqdm
import copy

In [ ]:
# --- Configurations ---
# Correctly specify the individual data directories
train_data_dir = '/content/train_images'
test_data_dir = '/content/test_images'
batch_size = 32
num_epochs = 10
learning_rate = 1e-4 # A smaller learning rate is safer for fine-tuning

# Dynamically determine the number of classes from the training dataset
# This assumes 'df' (the training dataframe) is already loaded and contains the 'label' column
num_classes = len(df['label'].unique()) # Set num_classes based on unique labels in the training dataframe

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
print(f"Number of classes detected: {num_classes}")

Using device: cuda
Number of classes detected: 22


### 3. Custom Dataset and Transforms
We define a `SkinDataset` class to handle image loading and label mapping. We also set up standard transformations, including resizing to 224x224 and normalization, to prepare images for the neural network.

In [ ]:
# --- Data Transformations ---
# Training set gets augmentation to prevent overfitting
train_transform = transforms.Compose([
    transforms.RandomResizedCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

# Test set only gets resized and normalized (NO augmentation)
test_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

### 4. Model Initialization
We use the `timm` library to load a pre-trained **ConvNeXt-Tiny** model. We adjust the final layer to match the number of disease classes in our dataset and initialize the AdamW optimizer.

In [ ]:
# Load pre-trained ConvNeXt and automatically adjust the final classification layer
model = timm.create_model('convnext_tiny', pretrained=True, num_classes=num_classes)
model = model.to(device)

# ConvNeXt trains best with the AdamW optimizer rather than standard Adam or SGD
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=1e-4)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model.safetensors:   0%|          | 0.00/114M [00:00<?, ?B/s]

In [ ]:
import os
import pandas as pd
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from tqdm import tqdm

# Define the custom dataset for unlabelled test images
class UnlabeledTestDataset(Dataset):
    def __init__(self, test_dir, transform=None):
        self.test_dir = test_dir
        # Grab all image files from the folder
        self.image_files = [f for f in os.listdir(test_dir) if f.endswith(('.png', '.jpg', '.jpeg'))]
        self.transform = transform

    def __len__(self):
        return len(self.image_files)

    def __getitem__(self, idx):
        # Get the image name and load the image
        img_name = self.image_files[idx]
        img_path = os.path.join(self.test_dir, img_name)
        image = Image.open(img_path).convert('RGB')

        if self.transform:
            image = self.transform(image)

        # Return BOTH the image tensor and the filename
        return image, img_name

### 5. Inference and Submission Generation
Using the trained model, we run predictions on the unlabeled test images. The results are mapped back to their respective labels and saved as `submission.csv`.

In [ ]:
# --- Configuration ---
# UPDATE THIS to the exact path of your unlabelled test images folder
test_images_folder = '/content/test_images'

# Create an inverse mapping from index to label name
# We use the dataset object created earlier which has the label_map
idx_to_label = {v: k for k, v in dataset.label_map.items()}

# Use the exact same test_transform from your training script
test_dataset = UnlabeledTestDataset(test_dir=test_images_folder, transform=test_transform)

# Batch size can be larger here since we aren't storing gradients
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False, num_workers=2)

# Ensure your model is in evaluation mode
model.eval()

# List to hold our final CSV rows
submission_results = []

print("Generating predictions for submission...")

# Run without calculating gradients to save memory
with torch.no_grad():
    for images, image_names in tqdm(test_loader, desc="Predicting"):
        images = images.to(device)

        # Get the model's raw outputs
        outputs = model(images)

        # Find the index of the highest probability (the predicted class)
        _, predictions = torch.max(outputs, 1)

        # Move predictions off the GPU and convert to a standard Python list
        predictions = predictions.cpu().numpy()

        # Match each prediction to its corresponding image filename
        for img_name, pred in zip(image_names, predictions):

            # Convert the numeric prediction to the string label
            label_name = idx_to_label[pred]

            img_id = img_name

            # Append the row to our results list
            submission_results.append({
                'image_id': img_id,
                'label': label_name
            })

# --- Create and Save the CSV ---
# Convert the list of dictionaries into a pandas DataFrame
submission_df = pd.DataFrame(submission_results)

# Save to CSV (index=False prevents pandas from adding an extra column of row numbers)
submission_df.to_csv('/content/submission.csv', index=False)

print("\nSuccess! Saved to /content/submission.csv")
display(submission_df.head()) # Preview the first 5 rows

Generating predictions for submission...


Predicting: 100%|██████████| 58/58 [00:33<00:00,  1.73it/s]


Success! Saved to /content/submission.csv


,image_id,label
0,d9a462c8-12c6-4a2b-a164-1b66b989aa45.jpeg,DrugEruption
1,62463bab-1f01-4649-90bd-aa8876af1bf6.jpeg,Vasculitis
2,a894e4a4-a05f-41be-bd17-6c0a6578098e.jpeg,DrugEruption
3,51cf4771-1de6-43ad-8258-165164869c2e.jpeg,DrugEruption
4,98eab9a0-cf86-4e72-a539-c0686a33904f.jpeg,Vasculitis


In [ ]:
import pandas as pd
import os

# --- DataLoaders ---
# Use the custom SkinDataset for the training dataset, which correctly handles the flat image directory structure
train_dataset = SkinDataset(df, transform=train_transform)

# Load test data and create file paths
test_df = pd.read_csv('/content/submission.csv')
test_image_dir = '/content/test_images'
if 'image_id' in test_df.columns:
    test_df['file_path'] = test_df['image_id'].apply(lambda x: os.path.join(test_image_dir, x))
else:
    print("Test CSV does not have an 'image_id' column. Please check.")

# Use the custom SkinDataset for the test dataset as well
test_dataset = SkinDataset(test_df, transform=test_transform)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)

In [ ]:
best_acc = 0.0
best_model_wts = copy.deepcopy(model.state_dict())

for epoch in range(num_epochs):
    print(f'\nEpoch {epoch+1}/{num_epochs}')
    print('-' * 10)

    # --- TRAINING PHASE ---
    model.train()
    running_loss = 0.0
    running_corrects = 0

    for inputs, labels in tqdm(train_loader, desc="Training"):
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad() # Clear old gradients

        # Forward pass
        outputs = model(inputs)
        _, preds = torch.max(outputs, 1)
        loss = criterion(outputs, labels)

        # Backward pass and optimize
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * inputs.size(0)
        running_corrects += torch.sum(preds == labels.data)

    epoch_loss = running_loss / len(train_dataset)
    epoch_acc = running_corrects.double() / len(train_dataset)
    print(f'Train Loss: {epoch_loss:.4f} Acc: {epoch_acc:.4f}')

    # --- VALIDATION (TEST) PHASE ---
    model.eval()
    val_loss = 0.0
    val_corrects = 0

    with torch.no_grad(): # No gradients needed for testing
        for inputs, labels in tqdm(test_loader, desc="Testing"):
            inputs, labels = inputs.to(device), labels.to(device)

            outputs = model(inputs)
            _, preds = torch.max(outputs, 1)
            loss = criterion(outputs, labels)

            val_loss += loss.item() * inputs.size(0)
            val_corrects += torch.sum(preds == labels.data)

    if len(test_dataset) > 0:
        val_epoch_loss = val_loss / len(test_dataset)
        val_epoch_acc = val_corrects.double() / len(test_dataset)
        print(f'Test Loss: {val_epoch_loss:.4f} Acc: {val_epoch_acc:.4f}')

        # --- SAVE THE BEST MODEL ---
        if val_epoch_acc > best_acc:
            print(f"*** New best model found! Accuracy improved from {best_acc:.4f} to {val_epoch_acc:.4f} ***")
            best_acc = val_epoch_acc
            best_model_wts = copy.deepcopy(model.state_dict())
            # Save to disk
            torch.save(model.state_dict(), 'best_convnext_model.pth')
    else:
        print('Test dataset is empty. Skipping validation metrics calculation and model saving for this epoch.')

print(f'\nTraining complete. Best Test Accuracy: {best_acc:.4f}')

# Load the best weights back into the model for future use
model.load_state_dict(best_model_wts)



Epoch 1/10
----------


Training: 100%|██████████| 271/271 [02:51<00:00,  1.58it/s]


Train Loss: 2.1115 Acc: 0.3607


Testing: 100%|██████████| 116/116 [00:36<00:00,  3.17it/s]


Test Loss: 4.8285 Acc: 0.1193
*** New best model found! Accuracy improved from 0.0000 to 0.1193 ***

Epoch 2/10
----------


...

Training: 100%|██████████| 271/271 [02:42<00:00,  1.67it/s]


Train Loss: 0.4101 Acc: 0.8675


Testing: 100%|██████████| 116/116 [00:37<00:00,  3.11it/s]

Test Loss: 9.4908 Acc: 0.0780

Training complete. Best Test Accuracy: 0.1193


<All keys matched successfully>

In [ ]:
import torch

# Method 1: Display the best accuracy recorded during training
print(f'Best Validation Accuracy achieved during training: {best_acc:.4f}')

# Method 2: Calculate accuracy on the training set to check for fit
model.eval()
correct = 0
total = 0

with torch.no_grad():
    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        outputs = model(inputs)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print(f'Final Accuracy on the training images: {100 * correct / total:.2f}%')

Best Validation Accuracy achieved during training: 0.1193
Final Accuracy on the training images: 46.34%


### 6. References
Below we visualize the class distribution from the training metadata and provide the source dataset reference.

**Kaggle Dataset:** https://www.kaggle.com/t/e12adecc3ffe4eeca1477e6765990587